# 03_build_indicators.py vs Spring Boot FinancialIndicatorService 대조 검증

PROJECT_INSTRUCTIONS.md 4.3 요구사항: 03단계 결과가 기존 Java 계산값과 일치하는지 검증.

**방법**: ValuePick 로컬 DB(`db-container`)에 우리 Parquet 원본을 역주입(`_seed_valuepick_db.py`) →
`/admin/indicator/calculate/{year}/{reprtCode}` API로 Java가 실제 계산 → `STOCK_INDICATOR` 테이블 결과를
JDBC로 읽어와 `data/indicators`(03번 결과)와 종목별로 diff 비교.

**사전 조건**: `docker-compose-local.yml`로 ValuePick(`db`, `bn`)이 떠 있어야 하고, `spark-master`가
`valuepick-local_custom-network`에 연결되어 있어야 함(`docker network connect`).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MYSQL_URL = "jdbc:mysql://db:3306/valuepick?serverTimezone=Asia/Seoul&characterEncoding=UTF-8"
MYSQL_PROPS = {"user": "valuepick", "password": "1234", "driver": "com.mysql.cj.jdbc.Driver"}

spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("verify_indicators") \
    .config("spark.jars", "/opt/spark/jars/mysql-connector-j-8.3.0.jar") \
    .getOrCreate()

ours = spark.read.parquet("/opt/spark-apps/data/indicators")
java = spark.read.jdbc(url=MYSQL_URL, table="STOCK_INDICATOR", properties=MYSQL_PROPS)

print(f"우리(03번) 결과: {ours.count()}건")
print(f"Java(STOCK_INDICATOR) 결과: {java.count()}건")

## 종목별 diff 계산
지표별로 |우리값 - Java값| 을 계산. 반올림 오차(소수점 둘째자리) 감안해 0.05 초과만 "불일치"로 표시.
momentum/f_score/eps_growth_rate는 Java가 서버 기동 시점(오늘 날짜) 기준으로 1개월전/12개월전을 찾아서
DB에 정확히 그 날짜 데이터가 없으면 NULL이 되므로, 값 존재 여부 자체가 다를 수 있음(별도 확인).

In [ ]:
metrics = ["eps", "bps", "per", "pbr", "roe", "debt_ratio", "dividend_yield", "roa"]
TOLERANCE = 0.05

ours_r = ours.select("stock_code", *[F.col(m).alias(f"{m}_ours") for m in metrics])
java_r = java.select("stock_code", *[F.col(m).alias(f"{m}_java") for m in metrics])

joined = ours_r.join(java_r, on="stock_code", how="inner")

diff_cols = []
for m in metrics:
    joined = joined.withColumn(f"{m}_diff", F.abs(F.col(f"{m}_ours") - F.col(f"{m}_java")))
    diff_cols.append(f"{m}_diff")

joined = joined.withColumn("max_diff", F.greatest(*[F.coalesce(F.col(c), F.lit(0.0)) for c in diff_cols]))

print(f"대조 대상(양쪽 다 있는 종목): {joined.count()}건")
mismatched = joined.filter(F.col("max_diff") > TOLERANCE)
print(f"불일치(오차 > {TOLERANCE}): {mismatched.count()}건")

## 불일치 종목 상세 (있다면 원인 규명 필요)

In [ ]:
cols_to_show = ["stock_code"] + [c for m in metrics for c in (f"{m}_ours", f"{m}_java", f"{m}_diff")]
mismatched.select(*cols_to_show).orderBy(F.desc("max_diff")).toPandas()

## momentum / f_score / eps_growth_rate 별도 확인
Java는 서버 기동 시점(오늘) 기준 1개월전/12개월전 데이터가 DB에 정확히 없으면 NULL. 우리는 01_ingest_raw.py가
실제로 그 시점 스냅샷을 수집했으므로 값이 있을 수 있음 - "다르다"가 아니라 "입력 데이터 커버리지 차이".

In [ ]:
extra_metrics = ["momentum", "f_score", "eps_growth_rate"]
ours_extra = ours.select("stock_code", *[F.col(m).alias(f"{m}_ours") for m in extra_metrics])
java_extra = java.select("stock_code", *[F.col(m).alias(f"{m}_java") for m in extra_metrics])
joined_extra = ours_extra.join(java_extra, on="stock_code", how="inner")

for m in extra_metrics:
    both_null = joined_extra.filter(F.col(f"{m}_ours").isNull() & F.col(f"{m}_java").isNull()).count()
    both_value = joined_extra.filter(F.col(f"{m}_ours").isNotNull() & F.col(f"{m}_java").isNotNull()).count()
    only_ours = joined_extra.filter(F.col(f"{m}_ours").isNotNull() & F.col(f"{m}_java").isNull()).count()
    only_java = joined_extra.filter(F.col(f"{m}_ours").isNull() & F.col(f"{m}_java").isNotNull()).count()
    print(f"{m}: 둘다 NULL={both_null}, 둘다 값있음={both_value}, 우리만 값있음={only_ours}, Java만 값있음={only_java}")